# Keskar et al. (2017), *On large-batch training for deep learning* — Figure 3, panels F1, C1, C2

Small-batch (256) vs large-batch (10% of the training set) ADAM training, then the cross-entropy loss and accuracy
along the line $\theta(\alpha)=\alpha\,\theta_{LB}+(1-\alpha)\,\theta_{SB}$, $\alpha\in[-1,2]$, on train and test data.

Code: `data.py`, `models.py`, `train.py`, `interpolate.py`, `plot_fig3.py`; `run_all.sh` runs everything
(about 3 h on an Apple M4 Pro, dominated by the VGG-like C2). Set `RUN_TRAINING = True` to retrain.

In [ ]:
import os, json, subprocess
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display
RUN_TRAINING = False
if RUN_TRAINING:
    subprocess.run(["bash", "run_all.sh"], check=True)

## Training curves of the SB and LB runs (cf. Figure 2 of the paper)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for ax, net in zip(axes, ["F1", "C1", "C2"]):
    for reg, c in [("SB", "b"), ("LB", "r")]:
        f = f"results/{net}_{reg}_s0.json"
        if not os.path.exists(f): continue
        d = json.load(open(f))
        ax.plot(d["epoch"], 100*np.array(d["train_acc"]), c+"--", label=f"{reg} train")
        ax.plot(d["epoch"], 100*np.array(d["test_acc"]), c+"-", label=f"{reg} test")
        print(f"{net} {reg}: batch {d['batch_size']}, {d['epoch'][-1]} epochs, train acc {100*d['train_acc'][-1]:.2f}%, test acc {100*d['test_acc'][-1]:.2f}%")
    ax.set_title(net); ax.set_xlabel("epoch"); ax.set_ylabel("accuracy (%)"); ax.legend(fontsize=8); ax.grid(ls=":")
plt.tight_layout(); plt.show()

## Figure 3 reproduced
Two ways of handling batch-norm statistics at the interpolated weights are shown: interpolating the running statistics
like every other tensor (`interp`), and re-estimating them on the training set at every $\alpha$ (`recompute`).

In [ ]:
!python3 plot_fig3.py --bn interp
!python3 plot_fig3.py --bn recompute
display(Image("figures/fig3_reproduced_bn-interp.png"))
display(Image("figures/fig3_reproduced_bn-recompute.png"))

## Numbers at the two end points

In [ ]:
for net in ["F1", "C1", "C2"]:
    f = f"results/{net}_interp_s0.json"
    if not os.path.exists(f): continue
    d = json.load(open(f)); a = np.array(d["alpha"])
    for mode in ["interp", "recompute"]:
        r = d[mode]; i0, i1 = np.argmin(abs(a)), np.argmin(abs(a-1))
        print(f"{net} bn={mode:9s} SB(alpha=0): train loss {r['train_loss'][i0]:.4f} test acc {100*r['test_acc'][i0]:.2f}% | "
              f"LB(alpha=1): train loss {r['train_loss'][i1]:.4f} test acc {100*r['test_acc'][i1]:.2f}%")